# Multi-Agent Research Pipeline: step by step

Run each cell in order to see what every agent produces. Kernel: **.venv** (Python 3.12).

Pipeline: **Search agent** → **Reader agent** → **Writer chain** → **Critic chain**

## 1. Tools
Test the tools on their own first. A tool is just a function the LLM can call, so you can call it yourself with `.invoke()`.

In [ ]:
from src.tools.tools import web_search, scrape_url

res = web_search.invoke({"query": "LangChain 1.0 release"})
[r["url"] for r in res["results"]]

In [ ]:
print(scrape_url.invoke({"url": "https://en.wikipedia.org/wiki/LangChain"})[:1000])

## 2. Search agent (LLM + `web_search` tool)
Stream it to watch the agent think: AI asks for a tool → tool answers → AI writes the summary.

In [ ]:
from src.agents.agents import build_search_agent

topic = "The impact of AI on the job market in 2026"

for state in build_search_agent().stream(
    {"messages": [("user", f"Find recent, reliable and detailed information about: {topic}")]},
    stream_mode="values",
):
    state["messages"][-1].pretty_print()

search = state["messages"][-1].content

## 3. Reader agent (LLM + `scrape_url` tool)
The pipeline's `reader_step` pulls the URLs out of the search summary and lets the agent pick one to scrape.

In [ ]:
from src.pipelines.pipeline import reader_step

reader = reader_step(topic, search)
print(reader)

## 4. Writer chain (`prompt | llm | parser`, no tools)
A chain runs in one fixed step: fill the prompt, call the LLM, get back a string.

In [ ]:
from IPython.display import Markdown
from src.pipelines.pipeline import writer_step

report = writer_step(topic, search, reader)
Markdown(report)

## 5. Critic chain

In [ ]:
from src.pipelines.pipeline import critic_step

Markdown(critic_step(report))

## 6. All at once
This is exactly what `main.py` runs.

In [ ]:
from src.pipelines.pipeline import run_research_pipeline

# results = run_research_pipeline("Future of LLMs in the tech industry")